# `expansion_hunter_03_budget` — pilot, cost, keep list

Run **in a VWB JupyterLab app** after a one-sample smoke in
[`expansion_hunter_02_run_batch.ipynb`](expansion_hunter_02_run_batch.ipynb).

This notebook does **not** replace `_02`. `_02` stays the one-sample smoke.
Here we:

1. Draw a random **pilot** (~100 samples) from the v9 CRAM manifest (or a CSV)
2. Submit **minicram then genotype** on native Google Batch (flags off until you flip them)
3. Estimate **p90 / p95** cost from VM wall time + Nearline `data_transfer_stats`
4. Write `keep.csv` under the run ledger so
   [`expansion_hunter_04_dispatch.ipynb`](expansion_hunter_04_dispatch.ipynb)
   can run the rest of the cohort under `BUDGET_USD`

Shared ledger: `scripts/vwb_batch/`. Sample outputs stay at
`gs://…/batchRuns/expansion_hunter/{sample_id}/`. Run metadata lives at
`gs://…/batchRuns/expansion_hunter/runs/{RUN_ID}/`.

Production catalog is the 711-locus degenerate panel. One-sample smoke on
`1000000` already SUCCEEDED (`eh-mini-260929-051752` ~8 min / 67.5 MiB,
`eh-gt-260929-053643` ~1 min). Batch has **no DAG**: this notebook polls
minicram, then you flip genotype. Success = Batch SUCCEEDED **and** GCS
objects exist. `SUBMIT_*` stay **off** until you turn one on.


In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "dispatch.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "dispatch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, cost, dispatch, gcs, poll, registry, submit  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")
REF_FA = os.environ.get(
    "EH_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "EH_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
CATALOG_KIND = "degenerate"
DEG_CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "candidate_EH_Loci.GRCh38.degenerate.json"
CATALOG = os.environ.get(
    "EH_CATALOG",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/expansion_hunter/candidate_EH_Loci.GRCh38.degenerate.json",
)
STAGE_CATALOG = True

PRINT_READS_DOCKER = os.environ.get(
    "EH_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
EH_DOCKER = os.environ.get(
    "EH_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-expansion-hunter:0.1.0",
)

OUT_PREFIX = os.environ.get(
    "EH_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
)
RUN_ID = os.environ.get("EH_RUN_ID") or f"eh-{datetime.now(timezone.utc).strftime('%y%m%d')}-pilot"  # pin this for dispatch
COHORT_CSV = os.environ.get("EH_COHORT_CSV", "")
PILOT_N = int(os.environ.get("EH_PILOT_N", "100"))
PILOT_SEED = int(os.environ.get("EH_PILOT_SEED", "1"))
SHARD_SIZE = int(os.environ.get("EH_SHARD_SIZE", "100"))
PARALLELISM = int(os.environ.get("EH_PILOT_PARALLELISM", "20"))
MAX_RUN_DURATION = os.environ.get("EH_MAX_RUN_DURATION", "28800s")
DEFAULT_SEX = "female"

SUBMIT_PILOT_MINICRAM = False
SUBMIT_PILOT_GENOTYPE = False
WRITE_KEEP = True
BUDGET_USD = float(os.environ.get("EH_BUDGET_USD", "0") or 0)
RETRY_MARGIN = float(os.environ.get("EH_RETRY_MARGIN", "1.15"))
COST_QUANTILE = float(os.environ.get("EH_COST_QUANTILE", "0.90"))

PATHS = registry.run_paths(
    output_bucket=OUTPUT_BUCKET_GS.rstrip("/"),
    pipeline="expansion_hunter",
    run_id=RUN_ID,
)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_ID:", RUN_ID)
print("ledger:", PATHS.prefix)
print("OUT_PREFIX:", OUT_PREFIX)
print("CATALOG:", CATALOG)
print("PILOT_N:", PILOT_N, "SEED:", PILOT_SEED)
print("PARALLELISM:", PARALLELISM, "SHARD_SIZE:", SHARD_SIZE)
print("BUDGET_USD:", BUDGET_USD, "quantile:", COST_QUANTILE, "margin:", RETRY_MARGIN)
print("SUBMIT_PILOT_MINICRAM:", SUBMIT_PILOT_MINICRAM, "SUBMIT_PILOT_GENOTYPE:", SUBMIT_PILOT_GENOTYPE)
print("WRITE_KEEP:", WRITE_KEEP)
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("gcloud:", shutil.which("gcloud"))


## PET SA / VPC

Jobs run as `PET_SA_EMAIL` on the workspace private VPC. No dsub. PET cannot
read Cloud Logging; use `{sample}.minicram.worker.log` / `{sample}.EH.worker.log`.


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
project, sa = alloc.require_vwb_env()
if not DEG_CATALOG_LOCAL.is_file():
    raise SystemExit(f"missing catalog: {DEG_CATALOG_LOCAL}")
print("project:", project)
print("PET_SA_EMAIL:", sa)
print("region:", REGION)
print("network: projects/{}/global/networks/network".format(project))


## Stage catalog

Copies the degenerate JSON to GCS if it is missing. Does not submit Batch.


In [ ]:
if STAGE_CATALOG and not gcs.exists(CATALOG):
    print("uploading", DEG_CATALOG_LOCAL, "->", CATALOG)
    gcs.upload_file(DEG_CATALOG_LOCAL, CATALOG)
else:
    print("catalog already on GCS" if gcs.exists(CATALOG) else "STAGE_CATALOG is off")
print("catalog records:", len(json.loads(DEG_CATALOG_LOCAL.read_text())))


## Cohort + pilot

Uses the v9 CRAM manifest (`person_id`, `cram_uri`, `cram_index_uri`) unless
`EH_COHORT_CSV` / `COHORT_CSV` points at a file with `batch.header.csv`
columns. Writes `cohort.csv` and `pilot.csv` to the run ledger (no Batch).


In [ ]:
project, sa = alloc.require_vwb_env()
if COHORT_CSV:
    src = Path(COHORT_CSV)
    if not src.is_file():
        raise SystemExit(f"COHORT_CSV not found: {src}")
    all_records = cohort.read_csv(src)
    for rec in all_records:
        rec.setdefault("ref_fa", REF_FA)
        rec.setdefault("ref_fai", REF_FAI)
        rec.setdefault("catalog", CATALOG)
        rec.setdefault("sex", DEFAULT_SEX)
else:
    manifest = cohort.find_v9_manifest()
    print("MANIFEST:", manifest or "(not found — set EH_CRAM_MANIFEST or EH_COHORT_CSV)")
    if manifest is None:
        raise SystemExit("no cohort: set EH_COHORT_CSV or mount the v9 CRAM manifest")
    all_records = cohort.records_from_manifest(
        manifest, ref_fa=REF_FA, ref_fai=REF_FAI, catalog=CATALOG, sex=DEFAULT_SEX
    )

print("cohort:", len(all_records))
pilot_records = cohort.sample_pilot(all_records, PILOT_N, PILOT_SEED)
print("pilot:", len(pilot_records), "ids:", [r["sample_id"] for r in pilot_records[:8]], "...")

local_cohort = SCRATCH / f"{RUN_ID}.cohort.csv"
local_pilot = SCRATCH / f"{RUN_ID}.pilot.csv"
cohort.write_csv(local_cohort, all_records)
cohort.write_csv(local_pilot, pilot_records)
gcs.upload_file(local_cohort, PATHS.cohort_csv)
gcs.upload_file(local_pilot, PATHS.pilot_csv)
registry.write_run_json(
    PATHS,
    {
        "kind": "pilot",
        "catalog": CATALOG,
        "out_prefix": OUT_PREFIX,
        "pilot_n": len(pilot_records),
        "pilot_seed": PILOT_SEED,
        "cohort_n": len(all_records),
        "parallelism": PARALLELISM,
        "max_run_duration": MAX_RUN_DURATION,
    },
)
registry.append_event(PATHS, "run_created", cohort_n=len(all_records), pilot_n=len(pilot_records))
print("cohort.csv:", PATHS.cohort_csv)
print("pilot.csv:", PATHS.pilot_csv)


## Dry-run job JSON

Always builds the first minicram shard (TASKS_TSV, no fat `taskEnvironments`).
Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
env_rows = eh.env_rows(pilot_records, stage="minicram", out_prefix=OUT_PREFIX, project=project)
prepared = dispatch.prepare_shard(
    paths=PATHS,
    stage="minicram",
    shard=0,
    records=pilot_records[: min(SHARD_SIZE, len(pilot_records))],
    env_rows=env_rows[: min(SHARD_SIZE, len(env_rows))],
    build_job=eh.build_job,
    image=PRINT_READS_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
    parallelism=PARALLELISM,
    max_run_duration=MAX_RUN_DURATION,
    scratch=SCRATCH,
)
group = prepared["job"]["taskGroups"][0]
print("job_id:", prepared["job_id"])
print("n_tasks:", prepared["n_tasks"], "json_bytes:", prepared["job_json_bytes"])
print("parallelism:", group.get("parallelism"))
print("TASKS_TSV:", group["taskSpec"]["environment"]["variables"].get("TASKS_TSV"))
print("taskEnvironments:" , "taskEnvironments" in group)
print("maxRunDuration:", group["taskSpec"]["maxRunDuration"])
print("CRAM stays gs:// (not in job JSON when using TASKS_TSV)")


## Submit pilot minicram

Set **only** `SUBMIT_PILOT_MINICRAM = True` in the config cell. Skips samples
that already have minicram objects. Caps parallelism. New job id every submit.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
pending, done = poll.pending_records(
    pilot_records,
    stage="minicram",
    out_prefix=OUT_PREFIX,
    expected=eh.expected_objects,
    listing=listing,
)
print("minicram done:", len(done), "pending:", len(pending))
inflight = poll.inflight_sample_ids(registry.load_events(PATHS), "minicram")
pending = [r for r in pending if r["sample_id"] not in inflight]
print("pending not in-flight:", len(pending), "in-flight:", len(inflight))

if not SUBMIT_PILOT_MINICRAM:
    print("SUBMIT_PILOT_MINICRAM is off; not submitting.")
elif not pending:
    print("nothing to submit")
else:
    for shard, chunk in cohort.shards(pending, SHARD_SIZE):
        rows = eh.env_rows(chunk, stage="minicram", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS,
            stage="minicram",
            shard=shard,
            records=chunk,
            env_rows=rows,
            build_job=eh.build_job,
            image=PRINT_READS_DOCKER,
            out_prefix=OUT_PREFIX,
            project=project,
            region=REGION,
            sa=sa,
            parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION,
            scratch=SCRATCH,
        )
        dispatch.maybe_submit(
            prepared=prepared,
            paths=PATHS,
            stage="minicram",
            shard=shard,
            project=project,
            region=REGION,
            do_submit=True,
        )


## Poll minicram

Safe to re-run. Updates `status.tsv` + ledger. Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
now = registry.utcnow()
status_rows = []
for ev in registry.load_events(PATHS):
    if ev.get("event") != "job_submitted" or ev.get("stage") != "minicram":
        continue
    job_id = ev["job_id"]
    sample_ids = ev.get("sample_ids") or []
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("skip", job_id, exc)
        continue
    registry.append_event(PATHS, "job_polled", stage="minicram", job_id=job_id, state=polled["state"])
    print(job_id, polled["state"], "tasks", len(polled["tasks"]))
    status_rows.extend(
        poll.reconcile_shard(
            sample_ids=sample_ids,
            stage="minicram",
            job_id=job_id,
            shard=int(ev.get("shard") or 0),
            poll=polled,
            expected=lambda sid, st: eh.expected_objects(sid, st, OUT_PREFIX),
            now=now,
            listing=listing,
        )
    )
for row in status_rows:
    if row.get("objects_ok") in {True, "True"}:
        uri = eh.transfer_stats_uri(row["sample_id"], OUT_PREFIX)
        if uri in listing or gcs.exists(uri):
            try:
                row["network_bytes"] = cost.parse_transfer_stats(gcs.cat(uri))
            except Exception as exc:
                print("stats", row["sample_id"], exc)
registry.write_status(PATHS, status_rows)
ok = sum(1 for r in status_rows if r.get("objects_ok") in {True, "True"})
print("status rows:", len(status_rows), "objects_ok:", ok)
print("status:", PATHS.status_tsv)


## Submit pilot genotype

Only samples whose **minicram objects exist**. Set
`SUBMIT_PILOT_GENOTYPE = True` after minicram is far enough along.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
mini_pending, mini_done = poll.pending_records(
    pilot_records,
    stage="minicram",
    out_prefix=OUT_PREFIX,
    expected=eh.expected_objects,
    listing=listing,
)
gt_pending, gt_done = poll.pending_records(
    mini_done,
    stage="genotype",
    out_prefix=OUT_PREFIX,
    expected=eh.expected_objects,
    listing=listing,
)
inflight = poll.inflight_sample_ids(registry.load_events(PATHS), "genotype")
gt_pending = [r for r in gt_pending if r["sample_id"] not in inflight]
print("minicram ready:", len(mini_done), "still minicram:", len(mini_pending))
print("genotype done:", len(gt_done), "pending:", len(gt_pending), "in-flight:", len(inflight))

if not SUBMIT_PILOT_GENOTYPE:
    print("SUBMIT_PILOT_GENOTYPE is off; not submitting.")
elif not gt_pending:
    print("nothing to submit")
else:
    for shard, chunk in cohort.shards(gt_pending, SHARD_SIZE):
        rows = eh.env_rows(chunk, stage="genotype", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS,
            stage="genotype",
            shard=shard,
            records=chunk,
            env_rows=rows,
            build_job=eh.build_job,
            image=EH_DOCKER,
            out_prefix=OUT_PREFIX,
            project=project,
            region=REGION,
            sa=sa,
            parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION,
            scratch=SCRATCH,
        )
        dispatch.maybe_submit(
            prepared=prepared,
            paths=PATHS,
            stage="genotype",
            shard=shard,
            project=project,
            region=REGION,
            do_submit=True,
        )


## Poll genotype

Safe to re-run. Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
now = registry.utcnow()
status_rows = registry.load_status(PATHS)
status_rows = [r for r in status_rows if r.get("stage") != "genotype"]
for ev in registry.load_events(PATHS):
    if ev.get("event") != "job_submitted" or ev.get("stage") != "genotype":
        continue
    job_id = ev["job_id"]
    sample_ids = ev.get("sample_ids") or []
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("skip", job_id, exc)
        continue
    registry.append_event(PATHS, "job_polled", stage="genotype", job_id=job_id, state=polled["state"])
    print(job_id, polled["state"], "tasks", len(polled["tasks"]))
    status_rows.extend(
        poll.reconcile_shard(
            sample_ids=sample_ids,
            stage="genotype",
            job_id=job_id,
            shard=int(ev.get("shard") or 0),
            poll=polled,
            expected=lambda sid, st: eh.expected_objects(sid, st, OUT_PREFIX),
            now=now,
            listing=listing,
        )
    )
registry.write_status(PATHS, status_rows)
ok = sum(1 for r in status_rows if r.get("stage") == "genotype" and r.get("objects_ok") in {True, "True", "true"})
print("genotype objects_ok:", ok)


## Cost + keep list

Planning unit is **p90** (not the mean) × `RETRY_MARGIN`. Nearline bytes come
from `{sample}.data_transfer_stats.tsv`. VM time comes from Batch task events.
Rates are us-central1 list prices in `scripts/vwb_batch/cost.py`, not invoices.

Set `BUDGET_USD` and `WRITE_KEEP = True` to upload `keep.csv`. The dispatch
notebook reads that object. Completed pilot samples are always kept.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
status = { (r.get("sample_id"), r.get("stage")): r for r in registry.load_status(PATHS) }
costs = []
for rec in pilot_records:
    sid = rec["sample_id"]
    mini_ok = all(u in listing for u in eh.expected_objects(sid, "minicram", OUT_PREFIX))
    gt_ok = all(u in listing for u in eh.expected_objects(sid, "genotype", OUT_PREFIX))
    if not mini_ok:
        continue
    stats_uri = eh.transfer_stats_uri(sid, OUT_PREFIX)
    try:
        network = cost.parse_transfer_stats(gcs.cat(stats_uri))
    except Exception as exc:
        print("skip stats", sid, exc)
        continue
    mini_sec = float(status.get((sid, "minicram"), {}).get("seconds") or 0) or 0.0
    gt_sec = float(status.get((sid, "genotype"), {}).get("seconds") or 0) or 0.0
    row = cost.sample_cost_usd(
        minicram_seconds=mini_sec,
        genotype_seconds=gt_sec,
        network_bytes=network,
    )
    row["sample_id"] = sid
    row["genotype_done"] = gt_ok
    costs.append(row)
    print(
        f"{sid} mini={mini_sec:.0f}s gt={gt_sec:.0f}s "
        f"GiB={row['network_bytes']/(1024**3):.3f} usd={row['total_usd']:.4f} gt_ok={gt_ok}"
    )

full = [c for c in costs if c["genotype_done"] and c["genotype_seconds"] > 0]
planning_src = full if len(full) >= max(5, PILOT_N // 10) else costs
if not planning_src:
    raise SystemExit("no completed minicram costs yet; poll after SUBMIT_PILOT_MINICRAM")
if len(full) < 5:
    print("WARN: using minicram-heavy costs; run genotype before trusting the keep list")

summary = cost.summarize_pilot(planning_src, q=COST_QUANTILE)
print("n_costed:", summary["n"])
print("mean:", round(summary["mean_usd"], 4), "p50:", round(summary["p50_usd"], 4))
print("p90:", round(summary["p90_usd"], 4), "p95:", round(summary["p95_usd"], 4), "max:", round(summary["max_usd"], 4))
print("planning (q=%s):" % summary["quantile"], round(summary["planning_usd"], 4))
unit = summary["planning_usd"] * RETRY_MARGIN
print("unit with margin:", round(unit, 4))

lines = ["sample_id\tminicram_seconds\tgenotype_seconds\tnetwork_bytes\ttotal_usd\tgenotype_done"]
for c in costs:
    lines.append(
        f"{c['sample_id']}\t{c['minicram_seconds']}\t{c['genotype_seconds']}\t"
        f"{int(c['network_bytes'])}\t{c['total_usd']:.6f}\t{c['genotype_done']}"
    )
gcs.upload_text(PATHS.cost_pilot_tsv, "\n".join(lines) + "\n")

done_ids = {c["sample_id"] for c in full} or {c["sample_id"] for c in costs}
done_keep = [r for r in pilot_records if r["sample_id"] in done_ids]
n_keep = cost.n_keep(
    budget_usd=BUDGET_USD,
    planning_usd=summary["planning_usd"],
    retry_margin=RETRY_MARGIN,
    n_available=len(all_records),
) if BUDGET_USD > 0 else len(done_keep)
n_keep = max(n_keep, len(done_keep))
pool = [r for r in all_records if r["sample_id"] not in {x["sample_id"] for x in done_keep}]
rng = random.Random(PILOT_SEED)
rng.shuffle(pool)
keep = done_keep + pool[: max(0, n_keep - len(done_keep))]
print("BUDGET_USD:", BUDGET_USD, "n_keep:", len(keep), "includes completed pilot:", len(done_keep))
print("est. spend at planning unit:", round(len(keep) * unit, 2))

if WRITE_KEEP and (BUDGET_USD > 0 or done_keep):
    local_keep = SCRATCH / f"{RUN_ID}.keep.csv"
    cohort.write_csv(local_keep, keep)
    gcs.upload_file(local_keep, PATHS.keep_csv)
    registry.append_event(
        PATHS,
        "keep_written",
        n_keep=len(keep),
        budget_usd=BUDGET_USD,
        planning_usd=summary["planning_usd"],
        retry_margin=RETRY_MARGIN,
    )
    print("keep.csv:", PATHS.keep_csv)
else:
    print("WRITE_KEEP is off or BUDGET_USD is 0 and no completed samples")

print("Next: expansion_hunter_04_dispatch with EH_RUN_ID=%s" % RUN_ID)
